# spatial / state-level

state-level weighted summaries, choropleths, then k-means on the state health profiles.

_Jesus_

In [ ]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



In [ ]:

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid", context="talk")

In [ ]:

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:

data_path = "/kaggle/input/datasets/ajenks/brfss-2020-2024-cleaned-and-weighted/brfss_2020_2024_pooled_eda.parquet"

df = pd.read_parquet(data_path)

print("Shape:", df.shape)
df.head()

In [ ]:

keep_cols = [
    "STATE_FIPS",
    "SURVEY_YEAR",
    "_LLCPWT_POOLED",
    "_LLCPWT",
    "RECENT_CHECKUP",
    "GOT_FLUSHOT",
    "HAS_EXERCISE",
    "HAS_PERSONAL_DOCTOR",
    "GOOD_HEALTH",
    "EVER_SMOKED",
    "_BMI5CAT",
    "HAS_DIABETES",
    "HAS_DEPRESSION"
]

available_cols = [col for col in keep_cols if col in df.columns]
missing_cols = [col for col in keep_cols if col not in df.columns]

print("Available columns:")
print(available_cols)

print("\nMissing columns:")
print(missing_cols)

df = df[available_cols].copy()

print("\nFiltered shape:", df.shape)
df.head()

In [ ]:

binary_vars = [
    "RECENT_CHECKUP",
    "GOT_FLUSHOT",
    "HAS_EXERCISE",
    "HAS_PERSONAL_DOCTOR",
    "GOOD_HEALTH",
    "EVER_SMOKED",
    "HAS_DIABETES",
    "HAS_DEPRESSION"
]

analysis_cols = [
    "STATE_FIPS",
    "SURVEY_YEAR",
    "_LLCPWT_POOLED",
    "_LLCPWT",
    "RECENT_CHECKUP",
    "GOT_FLUSHOT",
    "HAS_EXERCISE",
    "HAS_PERSONAL_DOCTOR",
    "GOOD_HEALTH",
    "EVER_SMOKED",
    "_BMI5CAT",
    "HAS_DIABETES",
    "HAS_DEPRESSION"
]

analysis_cols = [col for col in analysis_cols if col in df.columns]
df_analysis = df[analysis_cols].copy()

df_analysis = df_analysis.dropna(subset=["STATE_FIPS", "_LLCPWT_POOLED"]).copy()
df_analysis["STATE_FIPS"] = df_analysis["STATE_FIPS"].astype("Int64")
df_analysis["SURVEY_YEAR"] = df_analysis["SURVEY_YEAR"].astype("Int64")

print("Analysis shape:", df_analysis.shape)
df_analysis.head()

In [ ]:

state_lookup = {
    1: ("Alabama", "AL"),
    2: ("Alaska", "AK"),
    4: ("Arizona", "AZ"),
    5: ("Arkansas", "AR"),
    6: ("California", "CA"),
    8: ("Colorado", "CO"),
    9: ("Connecticut", "CT"),
    10: ("Delaware", "DE"),
    11: ("District of Columbia", "DC"),
    12: ("Florida", "FL"),
    13: ("Georgia", "GA"),
    15: ("Hawaii", "HI"),
    16: ("Idaho", "ID"),
    17: ("Illinois", "IL"),
    18: ("Indiana", "IN"),
    19: ("Iowa", "IA"),
    20: ("Kansas", "KS"),
    21: ("Kentucky", "KY"),
    22: ("Louisiana", "LA"),
    23: ("Maine", "ME"),
    24: ("Maryland", "MD"),
    25: ("Massachusetts", "MA"),
    26: ("Michigan", "MI"),
    27: ("Minnesota", "MN"),
    28: ("Mississippi", "MS"),
    29: ("Missouri", "MO"),
    30: ("Montana", "MT"),
    31: ("Nebraska", "NE"),
    32: ("Nevada", "NV"),
    33: ("New Hampshire", "NH"),
    34: ("New Jersey", "NJ"),
    35: ("New Mexico", "NM"),
    36: ("New York", "NY"),
    37: ("North Carolina", "NC"),
    38: ("North Dakota", "ND"),
    39: ("Ohio", "OH"),
    40: ("Oklahoma", "OK"),
    41: ("Oregon", "OR"),
    42: ("Pennsylvania", "PA"),
    44: ("Rhode Island", "RI"),
    45: ("South Carolina", "SC"),
    46: ("South Dakota", "SD"),
    47: ("Tennessee", "TN"),
    48: ("Texas", "TX"),
    49: ("Utah", "UT"),
    50: ("Vermont", "VT"),
    51: ("Virginia", "VA"),
    53: ("Washington", "WA"),
    54: ("West Virginia", "WV"),
    55: ("Wisconsin", "WI"),
    56: ("Wyoming", "WY"),
    66: ("Guam", "GU"),
    72: ("Puerto Rico", "PR"),
    78: ("Virgin Islands", "VI")
}

state_lookup_df = pd.DataFrame(
    [
        {"STATE_FIPS": fips, "STATE_NAME": name, "STATE_ABBR": abbr}
        for fips, (name, abbr) in state_lookup.items()
    ]
)

state_lookup_df.head()

In [ ]:

def weighted_mean(series, weights):
    valid = series.notna() & weights.notna()
    if valid.sum() == 0:
        return np.nan
    return np.average(series[valid], weights=weights[valid])

In [ ]:

state_vars = [
    "RECENT_CHECKUP",
    "GOT_FLUSHOT",
    "HAS_EXERCISE",
    "HAS_PERSONAL_DOCTOR",
    "GOOD_HEALTH",
    "EVER_SMOKED",
    "HAS_DIABETES",
    "HAS_DEPRESSION"
]

state_vars = [col for col in state_vars if col in df_analysis.columns]

state_summary = (
    df_analysis.groupby("STATE_FIPS")
    .apply(
        lambda g: pd.Series({
            var: weighted_mean(g[var], g["_LLCPWT_POOLED"])
            for var in state_vars
        })
    )
    .reset_index()
)

state_summary.head()

In [ ]:

state_summary["n_indicators"] = state_summary[state_vars].notna().sum(axis=1)

state_summary = state_summary.merge(state_lookup_df, on="STATE_FIPS", how="left")

state_summary = state_summary[
    ["STATE_FIPS", "STATE_NAME", "STATE_ABBR"] +
    [col for col in state_summary.columns if col not in ["STATE_FIPS", "STATE_NAME", "STATE_ABBR"]]
]

state_summary.head()

In [ ]:

print("State-level summary shape:", state_summary.shape)
state_summary.describe().T

In [ ]:

key_map_vars = [
    "RECENT_CHECKUP",
    "GOT_FLUSHOT",
    "HAS_EXERCISE",
    "GOOD_HEALTH"
]

key_map_vars = [col for col in key_map_vars if col in state_summary.columns]

state_summary[key_map_vars].describe().T

In [ ]:

for col in key_map_vars:
    print(f"\nTop 5 states for {col}:")
    print(
        state_summary[["STATE_NAME", col]]
        .dropna()
        .sort_values(by=col, ascending=False)
        .head(5)
        .to_string(index=False)
    )

    print(f"\nBottom 5 states for {col}:")
    print(
        state_summary[["STATE_NAME", col]]
        .dropna()
        .sort_values(by=col, ascending=True)
        .head(5)
        .to_string(index=False)
    )

In [ ]:

state_summary[["STATE_NAME"] + key_map_vars].head(10)

In [ ]:

map_df = state_summary.copy()

exclude_territories = ["Guam", "Puerto Rico", "Virgin Islands"]
map_df = map_df[~map_df["STATE_NAME"].isin(exclude_territories)].copy()

print("Map data shape:", map_df.shape)
print("States included in maps:", map_df["STATE_NAME"].nunique())

In [ ]:

map_df["STATE_ABBR"] = map_df["STATE_ABBR"].astype(str)

map_cols = ["STATE_FIPS", "STATE_NAME", "STATE_ABBR"] + key_map_vars
map_df[map_cols].head()

In [ ]:

for col in key_map_vars:
    print(f"{col}: min = {map_df[col].min():.3f}, max = {map_df[col].max():.3f}")

In [ ]:

def make_state_choropleth(data, value_col, title, subtitle, colorscale):
    fig = px.choropleth(
        data,
        locations="STATE_ABBR",
        locationmode="USA-states",
        color=value_col,
        scope="usa",
        color_continuous_scale=colorscale,
        hover_name="STATE_NAME",
        hover_data={
            "STATE_ABBR": False,
            value_col: ":.1%"
        },
        labels={value_col: title}
    )

    fig.update_layout(
        title={
            "text": f"{title} by State<br><sup>{subtitle}</sup>",
            "x": 0.5
        },
        margin={"r": 20, "t": 80, "l": 20, "b": 20}
    )
    return fig

In [ ]:

fig = make_state_choropleth(
    data=map_df,
    value_col="RECENT_CHECKUP",
    title="Routine Checkup Within the Past Year",
    subtitle="Weighted BRFSS pooled estimates, 2020 to 2024",
    colorscale="Blues"
)

fig.show()

**fig 1.** recent checkup share by state, pooled 2020-2024 (darker = more).

In [ ]:

fig = make_state_choropleth(
    data=map_df,
    value_col="GOT_FLUSHOT",
    title="Flu Shot in the Past 12 Months",
    subtitle="BRFSS pooled estimates, 2020 to 2024",
    colorscale="Purples"
)

fig.show()

**fig 2.** flu shot share by state, pooled 2020-2024.

In [ ]:

fig = make_state_choropleth(
    data=map_df,
    value_col="HAS_EXERCISE",
    title="Physical Activity in the Past 30 Days",
    subtitle="BRFSS pooled estimates, 2020 to 2024",
    colorscale="Greens"
)

fig.show()

**fig 3.** any exercise (past 30 days) by state, pooled.

In [ ]:

fig = make_state_choropleth(
    data=map_df,
    value_col="GOOD_HEALTH",
    title="Good Self-Reported Health",
    subtitle="BRFSS pooled estimates, 2020 to 2024",
    colorscale="Reds"
)

fig.show()

**fig 4.** good self-rated health by state, pooled.

In [ ]:

cluster_vars = [
    "RECENT_CHECKUP",
    "GOT_FLUSHOT",
    "HAS_EXERCISE",
    "HAS_PERSONAL_DOCTOR",
    "GOOD_HEALTH",
    "HAS_DIABETES",
    "HAS_DEPRESSION",
    "EVER_SMOKED"
]

cluster_vars = [col for col in cluster_vars if col in state_summary.columns]

cluster_df = state_summary[["STATE_FIPS", "STATE_NAME", "STATE_ABBR"] + cluster_vars].copy()

print("Clustering variables:")
print(cluster_vars)

cluster_df.head()

In [ ]:

cluster_features = cluster_df[cluster_vars].copy()

scaler = StandardScaler()
cluster_scaled = scaler.fit_transform(cluster_features)

cluster_scaled_df = pd.DataFrame(
    cluster_scaled,
    columns=cluster_vars,
    index=cluster_df.index
)

cluster_scaled_df.head()

In [ ]:

inertia_values = []
silhouette_values = []
k_values = range(2, 7)

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(cluster_scaled_df)

    inertia_values.append(kmeans.inertia_)
    silhouette_values.append(silhouette_score(cluster_scaled_df, labels))

cluster_eval = pd.DataFrame({
    "k": list(k_values),
    "inertia": inertia_values,
    "silhouette_score": silhouette_values
})

cluster_eval

In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(cluster_eval["k"], cluster_eval["inertia"], marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia")
plt.title("Elbow Plot for K-Means Clustering")
plt.show()

In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(cluster_eval["k"], cluster_eval["silhouette_score"], marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette score")
plt.title("Silhouette Scores for K-Means Clustering")
plt.show()

In [ ]:

final_k = 3

kmeans_final = KMeans(n_clusters=final_k, random_state=42, n_init=10)
cluster_df["cluster"] = kmeans_final.fit_predict(cluster_scaled_df)

cluster_df[["STATE_NAME", "cluster"]].head(10)

In [ ]:

cluster_df["cluster"].value_counts().sort_index()

In [ ]:

cluster_profile = (
    cluster_df.groupby("cluster")[cluster_vars]
    .mean()
    .multiply(100)
    .round(1)
)

cluster_profile

In [ ]:

cluster_profile = (
    cluster_df.groupby("cluster")[cluster_vars]
    .mean()
    .multiply(100)
    .round(1)
)

cluster_profile

In [ ]:

cluster_profile_long = (
    cluster_profile.reset_index()
    .melt(id_vars="cluster", var_name="indicator", value_name="value")
)

cluster_profile_long.head()

In [ ]:

plt.figure(figsize=(12, 6))
sns.barplot(data=cluster_profile_long, x="indicator", y="value", hue="cluster")
plt.xlabel("Indicator")
plt.ylabel("Average percentage")
plt.title("Average State Indicator Values by Cluster")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

In [ ]:

for c in sorted(cluster_df["cluster"].unique()):
    states_in_cluster = cluster_df.loc[
        cluster_df["cluster"] == c, "STATE_NAME"
    ].sort_values().tolist()
    print(f"\nCluster {c}:")
    print(states_in_cluster)

side-note: `HAS_HEALTH_PLAN` is only in 2020 & 2024, `COST_BARRIER` only in 2024 (per the eda files). pulling those separately so they don't distort the main pooled clustering.

In [ ]:

path_2020 = "/kaggle/input/datasets/ajenks/brfss-2020-2024-cleaned-and-weighted/brfss_2020_eda.parquet"
path_2024 = "/kaggle/input/datasets/ajenks/brfss-2020-2024-cleaned-and-weighted/brfss_2024_eda.parquet"

df_2020 = pd.read_parquet(path_2020)
df_2024 = pd.read_parquet(path_2024)

print("2020 shape:", df_2020.shape)
print("2024 shape:", df_2024.shape)

In [ ]:
# Pull just the fields needed for the health plan add-on.
health_plan_cols = ["STATE_FIPS", "SURVEY_YEAR", "_LLCPWT", "HAS_HEALTH_PLAN"]

health_plan_2020 = df_2020[[col for col in health_plan_cols if col in df_2020.columns]].copy()
health_plan_2024 = df_2024[[col for col in health_plan_cols if col in df_2024.columns]].copy()

df_health_plan = pd.concat([health_plan_2020, health_plan_2024], ignore_index=True)

print(df_health_plan.shape)
df_health_plan.head()

In [ ]:
# Pull the 2024-only fields needed for the cost barrier add-on.
cost_barrier_cols = ["STATE_FIPS", "SURVEY_YEAR", "_LLCPWT", "COST_BARRIER"]

df_cost_barrier = df_2024[[col for col in cost_barrier_cols if col in df_2024.columns]].copy()

print(df_cost_barrier.shape)
df_cost_barrier.head()

In [ ]:

df_health_plan = df_health_plan.dropna(subset=["STATE_FIPS", "_LLCPWT", "HAS_HEALTH_PLAN"]).copy()
df_health_plan["STATE_FIPS"] = df_health_plan["STATE_FIPS"].astype("Int64")
df_health_plan["SURVEY_YEAR"] = df_health_plan["SURVEY_YEAR"].astype("Int64")

df_cost_barrier = df_cost_barrier.dropna(subset=["STATE_FIPS", "_LLCPWT", "COST_BARRIER"]).copy()
df_cost_barrier["STATE_FIPS"] = df_cost_barrier["STATE_FIPS"].astype("Int64")

health_plan_summary = (
    df_health_plan.groupby(["SURVEY_YEAR", "STATE_FIPS"])
    .apply(
        lambda g: pd.Series({
            "HAS_HEALTH_PLAN": weighted_mean(g["HAS_HEALTH_PLAN"], g["_LLCPWT"])
        })
    )
    .reset_index()
    .merge(state_lookup_df, on="STATE_FIPS", how="left")
)

cost_barrier_summary = (
    df_cost_barrier.groupby("STATE_FIPS")
    .apply(
        lambda g: pd.Series({
            "COST_BARRIER": weighted_mean(g["COST_BARRIER"], g["_LLCPWT"])
        })
    )
    .reset_index()
    .merge(state_lookup_df, on="STATE_FIPS", how="left")
)

print("Health plan summary shape:", health_plan_summary.shape)
print("Cost barrier summary shape:", cost_barrier_summary.shape)

health_plan_summary.head()

In [ ]:

cost_barrier_map_df = cost_barrier_summary.copy()
cost_barrier_map_df = cost_barrier_map_df[
    ~cost_barrier_map_df["STATE_NAME"].isin(["Guam", "Puerto Rico", "Virgin Islands"])
].copy()

fig = make_state_choropleth(
    data=cost_barrier_map_df,
    value_col="COST_BARRIER",
    title="Could Not See a Doctor Due to Cost",
    subtitle="BRFSS 2024 estimates",
    colorscale="OrRd"
)

fig.show()

**fig 5.** could-not-see-doctor-due-to-cost by state, 2024 only.

In [ ]:

health_plan_compare = (
    health_plan_summary.pivot(
        index=["STATE_FIPS", "STATE_NAME", "STATE_ABBR"],
        columns="SURVEY_YEAR",
        values="HAS_HEALTH_PLAN"
    )
    .reset_index()
)

health_plan_compare.columns.name = None
health_plan_compare = health_plan_compare.rename(
    columns={2020: "HAS_HEALTH_PLAN_2020", 2024: "HAS_HEALTH_PLAN_2024"}
)

health_plan_compare.head(10)

limitations: descriptive only; clusters depend on indicators + scaling + k; some access vars only in subset of years.